# 04 — Horizon analysis

Casts west-facing rays from each observer and measures when the eclipsed Sun clears the reconstructed skyline.


In [ ]:
# Load the horizon-analysis modules, event window, rasters, and observer inputs.
import json

import numpy as np
import pandas as pd

from eclipse_viewshed.project import repository_root
PROJECT_ROOT = repository_root()

from eclipse_viewshed import aoi, horizon, solar, surface

INTERIM = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
EXTERNAL = PROJECT_ROOT / "data" / "external"
PROCESSED.mkdir(parents=True, exist_ok=True)

DSM_VRT = INTERIM / "dsm_1m.vrt"
DTM_VRT = INTERIM / "dtm_1m.vrt"
CLASSES = INTERIM / "classes_1m.tif"
for path in (DSM_VRT, DTM_VRT, CLASSES):
    if not path.exists():
        raise FileNotFoundError(
            f"{path} missing - run notebooks 02 and 03 first"
        )

# Load the event window produced by notebook 01 and sample the solar path.
RAY_M = aoi.DEFAULT_RAY_M
contact_data = json.loads(
    (PROCESSED / "01_eclipse_contacts.json").read_text()
)
C1_CEST = contact_data["first_contact_cest"]
MAX_CEST = contact_data["maximum_cest"]
C4_CEST = contact_data["last_contact_cest"]
ECLIPSE_WINDOW = (C1_CEST, C4_CEST)

track = solar.sun_track(step_seconds=5)
tmax = solar.at_time(track, MAX_CEST)


## Eye elevations


In [ ]:
# Resolve each observer's eye elevation from its configured vertical reference.
observers = pd.read_csv(EXTERNAL / "observers.csv")
observers[["x_l72", "y_l72"]] = observers.apply(
    lambda row: pd.Series(aoi.to_lambert72(row["lat"], row["lon"])),
    axis=1,
)

eye_rows = []
for _, observer in observers.iterrows():
    z_eye, base = horizon.resolve_eye_z(
        observer["x_l72"],
        observer["y_l72"],
        DSM_VRT,
        DTM_VRT,
        z_mode=observer["z_mode"],
        eye_height_m=observer["eye_height_m"],
        abs_eye_z_taw=observer.get("abs_eye_z_taw"),
    )
    eye_rows.append({
        "name": observer["name"],
        "z_mode": observer["z_mode"],
        "base_taw": round(base, 2),
        "eye_height_m": observer["eye_height_m"],
        "z_eye_taw": round(z_eye, 2),
    })

eye = pd.DataFrame(eye_rows)
observers["z_eye_taw"] = eye["z_eye_taw"]
eye


## Placement check


In [ ]:
# Measure the local surroundings and flag observer coordinates that are poorly placed.
rows = []
for _, o in observers.iterrows():
    s = horizon.describe_surroundings(
        o["x_l72"], o["y_l72"], DSM_VRT, DTM_VRT, CLASSES, radius_m=300)
    rows.append({
        "observer": o["name"],
        "ground_taw": s["ground_taw"],
        "nearest water (m)": s.get("nearest_water_m"),
        "water within 100 m": f"{s.get('water_within_100m_pct', 0):.0f}%",
        "nearest obstruction (m)": s.get("nearest_obstruction_m"),
        "tallest within 50 m": s["tallest_within_50m"],
        "tallest within 200 m": s["tallest_within_200m"],
        "at point": surface.CLASS_LABELS.get(s.get("class_at_point"), "?"),
    })

placement = pd.DataFrame(rows)

# Flag coordinates that are not close to the river.
ON_BANK_WATER_M = 60

placement["near the river"] = [
    (w is not None and w <= ON_BANK_WATER_M)
    for w in placement["nearest water (m)"]
]
placement


## Horizon profiles and visibility


In [ ]:
# Cast a horizon profile from each observer and calculate eclipse visibility.
profiles = {}
results = []

for _, observer in observers.iterrows():
    profile = horizon.compute_horizon(
        DSM_VRT,
        observer["x_l72"],
        observer["y_l72"],
        observer["z_eye_taw"],
        classes_path=CLASSES,
        exclude_classes=(surface.CLASS_WATER,),
        ray_m=RAY_M,
        name=observer["name"],
    )
    profiles[observer["name"]] = profile

    visibility = horizon.visibility(
        profile,
        track,
        max_eclipse_cest=MAX_CEST,
        eclipse_window_cest=ECLIPSE_WINDOW,
    )
    visibility["z_mode"] = observer["z_mode"]
    visibility["median_horizon_deg"] = round(
        float(np.median(profile.horizon_deg)), 2
    )
    visibility["max_horizon_deg"] = round(
        float(profile.horizon_deg.max()), 2
    )
    results.append(visibility)

results = pd.DataFrame(results)
results.sort_values("clearance_at_max", ascending=False)


## Obstruction classes


In [ ]:
# Summarize the vegetation, building, and unknown features controlling each skyline.
rows = []
for name, p in profiles.items():
    for code in (surface.CLASS_VEGETATION, surface.CLASS_BUILDING,
                 surface.CLASS_UNKNOWN):
        sel = p.class_code == code
        if not sel.any():
            continue
        h = p.horizon_deg[sel]
        rows.append({
            "observer": name,
            "class": surface.CLASS_LABELS[code],
            "bearings": int(sel.sum()),
            "median_deg": round(float(np.median(h)), 2),
            "max_deg": round(float(h.max()), 2),
            "above maximum": int((h > tmax["altitude_deg"]).sum()),
            "above 3 deg": int((h > 3.0).sum()),
        })

by_class = pd.DataFrame(rows)
by_class


## Outputs


In [ ]:
# Save the horizon profiles, visibility results, class summary, and analysis settings.
out_dir = PROCESSED / "horizon_profiles"
out_dir.mkdir(parents=True, exist_ok=True)
for old in out_dir.glob("*.csv"):
    old.unlink()

for name, p in profiles.items():
    slug = "_".join(name.lower().replace(",", "").split())
    p.to_frame().to_csv(out_dir / f"{slug}.csv", index=False)

results.to_csv(PROCESSED / "04_visibility.csv", index=False)
by_class.to_csv(PROCESSED / "04_horizon_by_class.csv", index=False)

summary = {
    "ray_m": RAY_M,
    "az_range": [solar.WEDGE_AZ_MIN, solar.WEDGE_AZ_MAX],
    "curvature_k": horizon.K_REFRACTION,
    "excluded_classes": ["water"],
    "max_eclipse": {"cest": contact_data["maximum_hhmm"],
                    "altitude_deg": round(float(tmax["altitude_deg"]), 2),
                    "azimuth_deg": round(float(tmax["azimuth_deg"]), 2)},
    "observers": len(profiles),
    "best_by_clearance": results.sort_values(
        "clearance_at_max", ascending=False).iloc[0]["name"],
}
(PROCESSED / "04_horizon_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print(f"\nprofiles -> {out_dir.relative_to(PROJECT_ROOT)}")
